# Milestone 3.5: Data Quality & Leakage Audit

This notebook summarizes the split integrity audit. We search for exact duplicate images (via MD5) and near-duplicate images (via Difference Hashing) leaking across our train, validation, and test splits.

## Notebook Outline:
1. **Audit Execution**: Run the CLI script to analyze splits.
2. **Summary Audit**: Load the JSON audit log and analyze leakage counts.
3. **Visual Leak Verification**: Display leaked image pairs side-by-side to verify the detector.
4. **Architectural Implications**: Address the warning and recommendations for Milestone 4.

## 1. Execute Data Leakage Audit CLI

We execute `scripts/run_data_quality_audit.py` to compile split checks.

In [ ]:
# Run audit CLI with a Hamming threshold of 10 bits
!python ../scripts/run_data_quality_audit.py \
    --dataset-dir ../data/processed/yolo_remapped \
    --crops-dir ../data/processed/crops/gt \
    --mapping-path ../configs/class_id_mapping.json \
    --image-threshold 10 \
    --output-dir ../reports/experiments/data_quality \
    --force

## 2. Load Leakage Summary JSON

Let's load the generated summary JSON to inspect the metrics.

In [ ]:
import json
from pathlib import Path

SUMMARY_PATH = Path("../reports/experiments/data_quality/data_quality_summary.json")
with open(SUMMARY_PATH, "r") as f:
    summary = json.load(f)

print("=== AUDIT RESULTS ===")
print("Disjointness Passed:", summary["disjointness_passed"])
print("Exact duplicate leak pairs:", summary["exact_duplicate_groups_found"])
print("Near duplicate leak pairs:", summary["near_duplicate_groups_found"])
print("Total leaked images:", summary["leaked_images_count"])
print("Total leaked crops:", summary["leaked_crops_count"])
print("Dataset integrity issues found:", summary["integrity"]["passed"] is False)

## 3. Visual Leak Verification

Let's display some sample leaked image pairs side-by-side to visually inspect whether they are indeed identical/near-identical shelf views.

In [ ]:
import cv2
import matplotlib.pyplot as plt

leaked_pairs = summary.get("leaked_pairs", [])
if leaked_pairs:
    # Visualize first 3 leaked pairs
    to_show = min(3, len(leaked_pairs))
    for idx in range(to_show):
        leak = leaked_pairs[idx]
        p1 = Path("..") / leak["file_1"]
        p2 = Path("..") / leak["file_2"]
        
        img1 = cv2.imread(str(p1))
        img2 = cv2.imread(str(p2))
        
        if img1 is not None and img2 is not None:
            img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
            img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
            
            fig, axes = plt.subplots(1, 2, figsize=(12, 6))
            axes[0].imshow(img1)
            axes[0].set_title(f"Split: {leak['split_1']}\n{p1.name}")
            axes[0].axis("off")
            
            axes[1].imshow(img2)
            axes[1].set_title(f"Split: {leak['split_2']}\n{p2.name} (Dist: {leak['hamming_distance']})")
            axes[1].axis("off")
            
            plt.tight_layout()
            plt.show()
else:
    print("No image leaks found to display.")

## 4. Architectural Implications

### Why We Must Fail the Audit
- **Metric Inflation**: In Milestone 4, building a crop embedding similarity index will retrieve crops from identical shelf layouts. If an image is in `train` and its duplicate/near-duplicate is in `val` or `test`, the KNN lookup will find exact matches of adjacent packages, inflating retrieval metrics.
- **Next Action**: Before running Milestone 4, the split mapping must be reconstructed by grouping duplicate/near-duplicate image sets into a single split or deduplicating the dataset completely.